# Patrón Creacional: Factory Method

## Introducción
El patrón Factory Method proporciona una interfaz para la creación de objetos en una superclase, pero permite a las subclases alterar el tipo de objetos que se crearán. Es muy útil cuando el sistema debe delegar la creación de objetos a subclases, permitiendo mayor flexibilidad y escalabilidad.

## Objetivos
- Comprender el propósito y la implementación del patrón Factory Method.
- Identificar cuándo es útil y cuándo evitarlo.
- Comparar la solución con y sin el patrón.

## Ejemplo de la vida real
**Contexto: App de Banco**
En una aplicación bancaria, el sistema puede necesitar crear diferentes tipos de cuentas (Ahorros, Corriente, Nómina) dependiendo del cliente. El Factory Method permite que cada tipo de cuenta se cree a través de una subclase especializada, facilitando la extensión y el mantenimiento del sistema.

**¿Dónde se usa en proyectos reales?**
En sistemas de pagos, creación de usuarios, procesamiento de documentos, sistemas de notificaciones, etc. Por ejemplo, en una app bancaria, la lógica para crear una cuenta de ahorros o una cuenta corriente puede variar y el Factory Method permite encapsular esa lógica en subclases.

## Sin patrón Factory Method (forma errónea)
El código cliente debe conocer las clases concretas que va a instanciar. Esto hace que el sistema sea rígido y difícil de mantener si se agregan nuevos tipos de cuentas.

In [1]:
import abc

class Transporte(abc.ABC):
    @abc.abstractmethod
    def entregar(self):
        ...

class Terrestre(Transporte):
    def entregar(self):
        print('Entrega por medio terrestre')

class Acuativo(Transporte):
    def entregar(self):
        print('Entrega por medio acuático')

# El cliente decide qué clase instanciar
transporte = Terrestre()
transporte.entregar()

Entrega por medio terrestre


## Con patrón Factory Method (forma correcta)
El cliente solo interactúa con la interfaz general y no necesita saber los detalles de cada tipo de cuenta. Esto facilita la extensión y el mantenimiento.

In [2]:
import abc

class Logistica(abc.ABC):
    @abc.abstractmethod
    def crear_transporte(self):
        ...

    def plan_entrega(self):
        transporte = self.crear_transporte()
        transporte.entregar()


class LogisticaCamion(Logistica):
    def crear_transporte(self):
        print("Compra de llantas")
        return Terrestre()


class LogisticaBicicleta(Logistica):
    def crear_transporte(self):
        return Terrestre()


class LogisticaBarco(Logistica):
    def crear_transporte(self):
        return Acuativo()


logistica = LogisticaBarco()
logistica.plan_entrega()

Entrega por medio acuático


## UML del patrón Factory Method
```plantuml
@startuml
abstract class Logistica {
    + crear_transporte()
    + plan_entrega()
}
class LogisticaCamion
class LogisticaBicicleta
class LogisticaBarco
Logistica <|-- LogisticaCamion
Logistica <|-- LogisticaBicicleta
Logistica <|-- LogisticaBarco
Logistica ..> Transporte
abstract class Transporte {
    + entregar()
}
class Terrestre
class Acuativo
Transporte <|-- Terrestre
Transporte <|-- Acuativo
@enduml
```

## Otro ejemplo de la vida real (sin logística): Sistema de Notificaciones
**Contexto:** una app (ecommerce, banco, red social) necesita avisar al usuario por distintos canales — Email, SMS, Push — cuando ocurre un evento (pedido confirmado, código de verificación, etc.). El canal a usar puede depender de la preferencia del usuario, del país, o de si tiene la app instalada.

Igual que con la logística, el problema es el mismo: el cliente no debería tener que conocer ni instanciar directamente cada clase concreta de notificación.

### Sin patrón Factory Method (forma errónea)
El cliente conoce y decide directamente qué clase concreta instanciar mediante un `if/elif`. Si mañana se agrega un canal WhatsApp, hay que tocar este código cliente.

In [3]:
class EmailNotificacion:
    def enviar(self, mensaje):
        print(f'Enviando EMAIL: "{mensaje}"')

class SMSNotificacion:
    def enviar(self, mensaje):
        print(f'Enviando SMS: "{mensaje}"')

class PushNotificacion:
    def enviar(self, mensaje):
        print(f'Enviando PUSH: "{mensaje}"')

# El cliente decide qué clase instanciar
canal = "sms"
if canal == "email":
    notificacion = EmailNotificacion()
elif canal == "sms":
    notificacion = SMSNotificacion()
else:
    notificacion = PushNotificacion()

notificacion.enviar("Tu pedido fue confirmado")

Enviando SMS: "Tu pedido fue confirmado"


### Con patrón Factory Method (forma correcta)
Se crea una jerarquía de "notificadores" (uno por canal), cada uno responsable de fabricar su propio tipo de notificación. El método de plantilla `notificar()` vive en la clase base y no cambia nunca; solo `crear_notificacion()` varía por subclase. Agregar WhatsApp es crear una clase nueva, sin tocar nada existente (principio Abierto/Cerrado).

In [4]:
import abc

class Notificador(abc.ABC):
    @abc.abstractmethod
    def crear_notificacion(self):
        ...

    def notificar(self, mensaje):
        notificacion = self.crear_notificacion()
        notificacion.enviar(mensaje)


class NotificadorEmail(Notificador):
    def crear_notificacion(self):
        return EmailNotificacion()


class NotificadorSMS(Notificador):
    def crear_notificacion(self):
        return SMSNotificacion()


class NotificadorPush(Notificador):
    def crear_notificacion(self):
        return PushNotificacion()


notificador = NotificadorPush()
notificador.notificar("Tu pedido fue confirmado")

Enviando PUSH: "Tu pedido fue confirmado"


### UML del ejemplo de Notificaciones
```plantuml
@startuml
abstract class Notificador {
    + crear_notificacion()
    + notificar(mensaje)
}
class NotificadorEmail
class NotificadorSMS
class NotificadorPush
Notificador <|-- NotificadorEmail
Notificador <|-- NotificadorSMS
Notificador <|-- NotificadorPush
Notificador ..> INotificacion

interface INotificacion {
    + enviar(mensaje)
}
class EmailNotificacion
class SMSNotificacion
class PushNotificacion
INotificacion <|.. EmailNotificacion
INotificacion <|.. SMSNotificacion
INotificacion <|.. PushNotificacion
@enduml
```

## ¿Dónde más se usa el Factory Method? (más allá de logística y notificaciones)
El patrón aparece en cualquier lugar donde el **tipo exacto de objeto a crear depende de una condición que solo se conoce en tiempo de ejecución** (configuración, plataforma, entrada del usuario, entorno):

- **Drivers de base de datos:** una capa de acceso a datos que crea la conexión concreta (MySQL, PostgreSQL, SQLite) según la configuración, exponiendo siempre la misma interfaz `Conexion`.
- **Parsers/lectores de documentos:** un `LectorDocumento` que crea un `ParserCSV`, `ParserJSON` o `ParserXML` según la extensión del archivo recibido.
- **Toolkits de interfaz gráfica multiplataforma:** frameworks como Qt o Swing crean el `Boton`, `Checkbox` o `Ventana` "nativo" del sistema operativo (Windows, macOS, Linux) sin que el código de la app lo sepa.
- **Pasarelas de pago:** un `ProcesadorPago` que fabrica el cliente concreto de Stripe, PayPal o MercadoPago según el método elegido por el usuario, exponiendo siempre `procesar(monto)`.
- **Loggers:** una fábrica que crea un `HandlerConsola`, `HandlerArchivo` o `HandlerRemoto` (ej. envío a un servicio como Datadog) según el entorno (desarrollo vs. producción).
- **Videojuegos:** una fábrica de enemigos o vehículos que crea el `Orco`, `Dragon` o `Esqueleto` según el nivel o el bioma, todos implementando la interfaz `Enemigo`.
- **Frameworks web / ORMs:** al crear formularios o campos de un modelo (`CampoTexto`, `CampoFecha`, `CampoSelect`) a partir de la definición declarativa de un modelo de datos.

La receta es siempre la misma: **una clase base define el "qué" (el algoritmo que usa el producto) y delega el "cómo se crea el producto" a un método que las subclases sobrescriben.**

## Actividad
Crea tu propio Factory Method para una aplicación de notificaciones (Email, SMS, Push).

---

## Explicación de conceptos clave
- **Desacoplamiento:** El Factory Method desacopla la lógica de creación de objetos del código cliente, permitiendo que el sistema sea más flexible y extensible.
- **Extensibilidad:** Si se necesita agregar un nuevo tipo de producto, solo se crea una nueva subclase sin modificar el código existente.
- **Aplicación en la vida real:** Muy útil en sistemas donde la creación de objetos depende de condiciones dinámicas o de la configuración del usuario.

## Conclusión
El patrón Factory Method es fundamental para construir sistemas escalables y mantenibles. Permite delegar la creación de objetos a subclases, facilitando la extensión y el mantenimiento del código. Es especialmente útil en aplicaciones bancarias, sistemas de pagos, y cualquier sistema donde la variedad de productos o servicios pueda crecer con el tiempo.